# Task D — Explainability and robustness analysis on the best model from Task C

This notebook extends the supervised-learning project from **Task C**.  
For reproducibility, it re-runs the same candidate-model selection logic used in Task C, identifies the best validation model, and then performs:

1. **One explainability view** using **permutation importance** on the selected best model.
2. **One robustness stress test** using a **limited-feature setting** based on the most important raw features.

The notebook keeps all evidence visible through figures, tables, and short written interpretations.

## Study hypothesis

**Hypothesis:** A small number of raw traffic features dominate the model's decisions, so a model retrained using only the top 5 most important raw features should retain a substantial portion of the baseline performance, although some degradation is expected.

This is a reasonable hypothesis for cybersecurity tabular data because network-traffic metadata such as ports, packet length, protocol, and anomaly score often carry a large part of the predictive signal.

## Task / Question
**Import the required libraries, define a clean plotting style, and set the random seed so the analysis is reproducible.**

In [ ]:

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Task / Question
**Load the dataset, remove duplicate rows, engineer safe time features from the timestamp, and define the same leakage-aware feature set used in Task C.**

In [ ]:

candidate_paths = [
    Path("cybersecurity_attacks.csv"),
    Path("/mnt/data/cybersecurity_attacks.csv")
]

for path in candidate_paths:
    if path.exists():
        DATA_PATH = path
        break
else:
    raise FileNotFoundError("Could not find 'cybersecurity_attacks.csv'. Put the CSV next to the notebook or update DATA_PATH.")

df = pd.read_csv(DATA_PATH).drop_duplicates().copy()

df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df["Hour"] = df["Timestamp"].dt.hour
df["DayOfWeek"] = df["Timestamp"].dt.dayofweek
df["Month"] = df["Timestamp"].dt.month
df["Year"] = df["Timestamp"].dt.year

TARGET = "Attack Type"

selected_features = [
    "Source Port",
    "Destination Port",
    "Protocol",
    "Packet Length",
    "Packet Type",
    "Traffic Type",
    "Anomaly Scores",
    "Network Segment",
    "Log Source",
    "Hour",
    "DayOfWeek",
    "Month",
    "Year"
]

X = df[selected_features].copy()
y = df[TARGET].copy()

dataset_summary = pd.DataFrame({
    "Item": ["Rows after duplicate removal", "Number of selected features", "Target classes"],
    "Value": [len(df), len(selected_features), y.nunique()]
})

display(dataset_summary)
display(y.value_counts().rename_axis("Class").reset_index(name="Count"))

## Task / Question
**Create train, validation, and test splits with stratification. The validation split will be used for model comparison and explainability. The test split remains untouched until the end.**

In [ ]:

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Rows": [len(X_train), len(X_val), len(X_test)],
    "Fraction of full data": [
        len(X_train) / len(X),
        len(X_val) / len(X),
        len(X_test) / len(X)
    ]
})

display(split_summary.round(4))

## Task / Question
**Rebuild the Task C candidate-model comparison so the notebook can identify the best model in a self-contained way.**

In [ ]:

categorical_features = [c for c in selected_features if X[c].dtype == "object"]
numeric_features = [c for c in selected_features if c not in categorical_features]

linear_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), categorical_features)
    ]
)

model_spaces = {
    "SGD Logistic": {
        "preprocessor": linear_preprocessor,
        "base_model": SGDClassifier(
            loss="log_loss",
            penalty="l2",
            max_iter=1500,
            tol=1e-3,
            random_state=RANDOM_STATE
        ),
        "param_grid": {"alpha": [1e-4, 1e-3, 1e-2]}
    },
    "Decision Tree": {
        "preprocessor": tree_preprocessor,
        "base_model": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "param_grid": {"max_depth": [6, 12, None], "min_samples_leaf": [1, 5, 20]}
    },
    "Random Forest": {
        "preprocessor": tree_preprocessor,
        "base_model": RandomForestClassifier(
            n_estimators=120,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "param_grid": {"max_depth": [8, 14, None], "min_samples_leaf": [1, 5]}
    }
}

def build_pipeline(preprocessor, model):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

tuning_rows = []
best_models = {}

for model_name, config in model_spaces.items():
    best_score = -np.inf
    best_pipeline = None
    best_params = None

    for params in ParameterGrid(config["param_grid"]):
        model = clone(config["base_model"]).set_params(**params)
        pipeline = build_pipeline(config["preprocessor"], model)

        pipeline.fit(X_train, y_train)
        val_pred = pipeline.predict(X_val)

        macro_f1 = f1_score(y_val, val_pred, average="macro")
        row = {
            "Model": model_name,
            "Hyperparameters": params,
            "Validation Accuracy": accuracy_score(y_val, val_pred),
            "Validation Balanced Accuracy": balanced_accuracy_score(y_val, val_pred),
            "Validation Macro F1": macro_f1
        }
        tuning_rows.append(row)

        if macro_f1 > best_score:
            best_score = macro_f1
            best_pipeline = pipeline
            best_params = params

    best_models[model_name] = {
        "pipeline": best_pipeline,
        "best_params": best_params,
        "best_validation_macro_f1": best_score
    }

tuning_results = pd.DataFrame(tuning_rows).sort_values(
    by=["Validation Macro F1", "Validation Balanced Accuracy", "Validation Accuracy"],
    ascending=False
).reset_index(drop=True)

model_comparison = tuning_results.groupby("Model", as_index=False).first().sort_values(
    by="Validation Macro F1", ascending=False
).reset_index(drop=True)

display(model_comparison)

## Task / Question
**Visualize the validation-set model comparison and identify the best model to carry into Task D.**

In [ ]:

plot_df = model_comparison.melt(
    id_vars="Model",
    value_vars=["Validation Accuracy", "Validation Balanced Accuracy", "Validation Macro F1"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric", palette="Set2")
plt.ylim(0, 1)
plt.title("Task C model comparison reproduced for Task D")
plt.xlabel("")
plt.ylabel("Validation score")
plt.legend(title="")
plt.tight_layout()
plt.show()

best_model_name = model_comparison.loc[0, "Model"]
best_config = model_spaces[best_model_name]
best_params = best_models[best_model_name]["best_params"]

best_train_pipeline = best_models[best_model_name]["pipeline"]

print(f"Best validation model selected for Task D: {best_model_name}")
print(f"Best hyperparameters: {best_params}")

## Task / Question
**Refit the best model on Train + Validation and evaluate the baseline result on the untouched Test split. This is the baseline that the robustness stress test will be compared against.**

In [ ]:

final_model = build_pipeline(
    best_config["preprocessor"],
    clone(best_config["base_model"]).set_params(**best_params)
)

final_model.fit(X_train_val, y_train_val)
test_pred = final_model.predict(X_test)

baseline_metrics = {
    "Accuracy": accuracy_score(y_test, test_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, test_pred),
    "Macro F1": f1_score(y_test, test_pred, average="macro")
}

baseline_table = pd.DataFrame({
    "Metric": list(baseline_metrics.keys()),
    "Baseline Test Score": list(baseline_metrics.values())
})

display(baseline_table.round(4))

label_order = sorted(y.unique())
cm = confusion_matrix(y_test, test_pred, labels=label_order)

plt.figure(figsize=(6.5, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    xticklabels=label_order,
    yticklabels=label_order
)
plt.title(f"Baseline test confusion matrix — {best_model_name}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

## Task / Question
**Produce one explanation view using permutation importance on the validation split.**

Why this method is justified:
- it works with any fitted scikit-learn pipeline,
- it explains importance at the level of the original raw input columns,
- it is easy to interpret in a report.

To keep the run time practical, the code uses a capped validation sample if the validation set is very large.

In [ ]:

max_explain_rows = 2000
if len(X_val) > max_explain_rows:
    X_val_explain = X_val.sample(n=max_explain_rows, random_state=RANDOM_STATE)
    y_val_explain = y_val.loc[X_val_explain.index]
else:
    X_val_explain = X_val.copy()
    y_val_explain = y_val.copy()

perm = permutation_importance(
    best_train_pipeline,
    X_val_explain,
    y_val_explain,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="f1_macro",
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_val_explain.columns,
    "Mean importance": perm.importances_mean,
    "Std importance": perm.importances_std
}).sort_values("Mean importance", ascending=False).reset_index(drop=True)

display(importance_df.round(4))

## Task / Question
**Plot the explanation result so the most influential features are easy to interpret visually.**

In [ ]:

plt.figure(figsize=(9, 6))
sns.barplot(
    data=importance_df,
    y="Feature",
    x="Mean importance",
    palette="viridis"
)
plt.title(f"Permutation importance on validation data — {best_model_name}")
plt.xlabel("Mean decrease in macro-F1 after shuffling")
plt.ylabel("")
plt.tight_layout()
plt.show()

## Short interpretation of the explanation view
The top rows of the table and the bar plot above show which raw traffic features matter most for the selected model.  
If only a small number of features stand out clearly, that supports the idea that the model is relying on a compact subset of the available traffic information.

## Task / Question
**Run one robustness stress test using a limited-feature setting.**

Stress condition:
- select the **top 5 raw features** from the explanation analysis,
- retrain the **same model family with the same tuned hyperparameters** using only those features,
- compare the stressed model against the baseline on the same untouched test set.

This stress test is well aligned with the hypothesis because it checks whether the model truly depends on only a small subset of features.

In [ ]:

TOP_K = 5
top_features = importance_df["Feature"].head(TOP_K).tolist()

X_train_val_top = X_train_val[top_features].copy()
X_test_top = X_test[top_features].copy()

top_categorical = [c for c in top_features if X_train_val_top[c].dtype == "object"]
top_numeric = [c for c in top_features if c not in top_categorical]

linear_preprocessor_top = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), top_numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), top_categorical)
    ]
)

tree_preprocessor_top = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), top_numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), top_categorical)
    ]
)

top_preprocessor = linear_preprocessor_top if best_model_name == "SGD Logistic" else tree_preprocessor_top

stressed_model = build_pipeline(
    top_preprocessor,
    clone(best_config["base_model"]).set_params(**best_params)
)

stressed_model.fit(X_train_val_top, y_train_val)
stressed_pred = stressed_model.predict(X_test_top)

stressed_metrics = {
    "Accuracy": accuracy_score(y_test, stressed_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, stressed_pred),
    "Macro F1": f1_score(y_test, stressed_pred, average="macro")
}

display(pd.DataFrame({"Top 5 features used in stress test": top_features}))

## Task / Question
**Compare the baseline and stressed results in one compact table, including metric drops.**

In [ ]:

comparison_table = pd.DataFrame({
    "Metric": ["Accuracy", "Balanced Accuracy", "Macro F1"],
    "Baseline": [
        baseline_metrics["Accuracy"],
        baseline_metrics["Balanced Accuracy"],
        baseline_metrics["Macro F1"]
    ],
    "Stressed (top-5 features only)": [
        stressed_metrics["Accuracy"],
        stressed_metrics["Balanced Accuracy"],
        stressed_metrics["Macro F1"]
    ]
})

comparison_table["Absolute Drop"] = comparison_table["Baseline"] - comparison_table["Stressed (top-5 features only)"]

display(comparison_table.round(4))

## Task / Question
**Visualize the baseline-vs-stressed comparison so the robustness effect is easy to inspect.**

In [ ]:

plot_compare = comparison_table.melt(
    id_vars="Metric",
    value_vars=["Baseline", "Stressed (top-5 features only)"],
    var_name="Setting",
    value_name="Score"
)

plt.figure(figsize=(9, 5))
sns.barplot(data=plot_compare, x="Metric", y="Score", hue="Setting", palette="coolwarm")
plt.ylim(0, 1)
plt.title("Baseline vs stressed robustness comparison")
plt.ylabel("Test score")
plt.xlabel("")
plt.legend(title="")
plt.tight_layout()
plt.show()

## Task / Question
**Summarize the study in the required final table with the columns: Hypothesis, Evidence observed, Supported or not supported?, What I learned.**

In [ ]:

max_drop = comparison_table["Absolute Drop"].max()
avg_drop = comparison_table["Absolute Drop"].mean()

top_feature_text = ", ".join(top_features[:3])

if avg_drop <= 0.03:
    support_label = "Supported"
    evidence_sentence = (
        f"The explanation view showed concentrated importance in features such as {top_feature_text}. "
        f"When the model was retrained with only the top 5 features, the average metric drop was {avg_drop:.4f}, "
        f"which is small enough to suggest that a compact subset of features carries much of the signal."
    )
    learned_sentence = (
        "The model appears to rely heavily on a small core of traffic features. "
        "This is useful for simplification, faster deployment, and feature-priority discussions."
    )
else:
    support_label = "Partly supported / not fully supported"
    evidence_sentence = (
        f"The explanation view highlighted features such as {top_feature_text}, but the stressed model lost noticeable performance. "
        f"The average metric drop was {avg_drop:.4f} and the largest single drop was {max_drop:.4f}, "
        f"which suggests that the model still benefits from a broader feature set."
    )
    learned_sentence = (
        "A few features are important, but the full model does not depend on only those features. "
        "Secondary features still contribute meaningful complementary information."
    )

final_learning_table = pd.DataFrame([{
    "Hypothesis": "A small number of raw traffic features dominate the decisions, so a top-5-feature model should retain much of the baseline performance.",
    "Evidence observed": evidence_sentence,
    "Supported or not supported?": support_label,
    "What I learned": learned_sentence
}])

display(final_learning_table)

## Final takeaway
This study linked **explainability** and **robustness** in a single report:

- the best model from Task C was reproduced in a visible way,
- an explanation view showed which raw traffic features mattered most,
- a stress test checked whether the model remained effective when restricted to only those top features,
- the final summary table converted the results into a clear conclusion.

That combination helps move from “which model performs best?” to “why does it work, and how fragile is it under a realistic constraint?”